# 🇳🇬 NAFDAC Greenbook Scraper
### Scrape all Active Ingredients from greenbook.nafdac.gov.ng → Save as CSV

**Source:** https://greenbook.nafdac.gov.ng/ingredients  
**Pages:** 45 pages · ~50 ingredients per page · ~2,200+ total ingredients  
**Output:** `nafdac_ingredients.csv` — saved locally and optionally to Google Drive  

---
### What this notebook does:
1. Scrapes all 45 pages of NAFDAC active ingredients
2. Extracts: ingredient name, ingredient ID, NAFDAC URL, individual drug components
3. Cleans and deduplicates
4. Saves to CSV — locally in Colab AND optionally to your Google Drive

> **Run time:** ~60–90 seconds with polite 1s delay between pages

---
## Step 1 — Install & Import

In [9]:
!pip install -q requests beautifulsoup4 lxml

In [2]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

print('✅ Libraries ready.')

✅ Libraries ready.


---
## Step 2 — Scrape All 45 Pages

In [3]:
# ── Config ────────────────────────────────────────────────────────────────────

BASE_URL    = 'https://greenbook.nafdac.gov.ng'
START_URL   = f'{BASE_URL}/ingredients'
TOTAL_PAGES = 45
DELAY       = 1.0   # seconds between requests — be polite to the server

HEADERS = {
    'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/120.0.0.0 Safari/537.36',
    'Accept'         : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer'        : 'https://greenbook.nafdac.gov.ng/',
    'Connection'     : 'keep-alive',
}

print(f'Target  : {START_URL}')
print(f'Pages   : {TOTAL_PAGES}')
print(f'Delay   : {DELAY}s per page')
print(f'Est. time: ~{int(TOTAL_PAGES * DELAY + 30)}s')

Target  : https://greenbook.nafdac.gov.ng/ingredients
Pages   : 45
Delay   : 1.0s per page
Est. time: ~75s


In [4]:
# ── Scraper function ──────────────────────────────────────────────────────────

def scrape_page(page_num, session):
    """
    Scrape one page of NAFDAC ingredients.

    Page 1 = BASE_URL/ingredients (no ?page param)
    Page 2+ = BASE_URL/ingredients?page=N

    Returns list of dicts with keys:
        ingredient_name, ingredient_id, nafdac_url, page_scraped
    """
    url = START_URL if page_num == 1 else f'{START_URL}?page={page_num}'

    try:
        resp = session.get(url, headers=HEADERS, timeout=20)

        if resp.status_code == 404:
            return None, '404'
        if resp.status_code != 200:
            return [], f'HTTP {resp.status_code}'

        soup  = BeautifulSoup(resp.text, 'lxml')
        links = soup.find_all(
            'a',
            href=lambda h: h and '/ingredient/products/' in h
        )

        records = []
        for link in links:
            name = link.get_text(strip=True)
            href = link.get('href', '')

            # Extract numeric ingredient ID from URL
            id_match = re.search(r'/(\d+)$', href)
            if not id_match or not name:
                continue

            ingredient_id  = int(id_match.group(1))
            nafdac_url     = href if href.startswith('http') else BASE_URL + href

            records.append({
                'ingredient_name': name.strip(),
                'ingredient_id'  : ingredient_id,
                'nafdac_url'     : nafdac_url,
                'page_scraped'   : page_num
            })

        return records, 'ok'

    except requests.exceptions.Timeout:
        return [], 'timeout'
    except requests.exceptions.ConnectionError:
        return [], 'connection_error'
    except Exception as e:
        return [], str(e)


print('✅ Scraper function defined.')

✅ Scraper function defined.


In [5]:
# ── Run the scraper ───────────────────────────────────────────────────────────

all_records  = []
failed_pages = []
page_summary = []

# Use a Session to reuse TCP connections — faster and more reliable
session = requests.Session()

print('🔍 Starting NAFDAC Greenbook scrape...\n')

for page in tqdm(range(1, TOTAL_PAGES + 1), desc='Pages', unit='page'):

    records, status = scrape_page(page, session)

    if status == '404':
        print(f'\n  ⚠️  Page {page}: 404 — site may have fewer pages than expected. Stopping.')
        break

    if status != 'ok' or not records:
        print(f'\n  ❌ Page {page}: {status} — retrying once after 3s...')
        time.sleep(3)
        records, status = scrape_page(page, session)  # one retry
        if status != 'ok' or not records:
            print(f'     Still failed. Skipping page {page}.')
            failed_pages.append(page)
            page_summary.append({'page': page, 'count': 0, 'status': status})
            time.sleep(DELAY)
            continue

    all_records.extend(records)
    page_summary.append({'page': page, 'count': len(records), 'status': 'ok'})
    time.sleep(DELAY)

session.close()

print(f'\n✅ Scrape complete!')
print(f'   Total records collected : {len(all_records):,}')
print(f'   Pages scraped           : {len([p for p in page_summary if p["status"] == "ok"])}/{TOTAL_PAGES}')
print(f'   Failed pages            : {failed_pages if failed_pages else "None"}')

🔍 Starting NAFDAC Greenbook scrape...



Pages: 100%|██████████| 45/45 [01:04<00:00,  1.44s/page]


✅ Scrape complete!
   Total records collected : 2,221
   Pages scraped           : 45/45
   Failed pages            : None


---
## Step 3 — Clean & Deduplicate

In [6]:
# ── Build raw DataFrame ───────────────────────────────────────────────────────

raw_df = pd.DataFrame(all_records)

print(f'Raw records   : {len(raw_df):,}')
print(f'Columns       : {list(raw_df.columns)}')
print()
raw_df.head(10)

Raw records   : 2,221
Columns       : ['ingredient_name', 'ingredient_id', 'nafdac_url', 'page_scraped']



,ingredient_name,ingredient_id,nafdac_url,page_scraped
0,"2,4-Dichlorobenzyl alcohol , Amylmetacresol ,V...",2198,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
1,Abacavir Sulfate,585,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
2,Abacavir Sulfate; Lamivudine,1850,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
3,"ABACAVIR, DOLUTEGRAVIR, LAMIVUDINE",2335,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
4,Abacavir; Lamivudine,601,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
5,Abacavir; Lamivudine; Efavirenz,1467,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
6,Abacavir; Lamivudine; Zidovudine,602,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
7,Abiraterone Acetate,1377,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
8,Acarbose,636,https://greenbook.nafdac.gov.ng/ingredient/pro...,1
9,Aceclofenac,74,https://greenbook.nafdac.gov.ng/ingredient/pro...,1


In [7]:
# ── Clean & expand ingredient names ──────────────────────────────────────────

def clean_name(name):
    """Standardise ingredient name: strip whitespace, fix spacing around delimiters."""
    name = name.strip()
    # Normalise separators: semicolons and commas → consistent '; '
    name = re.sub(r'\s*;\s*', '; ', name)
    name = re.sub(r'\s*,\s*', ', ', name)
    # Remove double spaces
    name = re.sub(r'  +', ' ', name)
    return name

def extract_components(name):
    """
    Split combination drug name into individual active ingredient components.
    e.g. 'Abacavir; Lamivudine; Efavirenz' → ['Abacavir', 'Lamivudine', 'Efavirenz']
    """
    parts = re.split(r'[;,/+]', name)
    components = []
    for p in parts:
        p = p.strip()
        # Remove salt/ester suffixes for the generic name
        p_generic = re.sub(
            r'\b(sulfate|hydrochloride|hcl|sodium|potassium|acetate|phosphate|'
            r'maleate|fumarate|tartrate|citrate|mesylate|tosylate|besylate|'
            r'monohydrate|dihydrate|anhydrous)\b',
            '', p, flags=re.IGNORECASE
        ).strip()
        # Remove bracketed notes like '(Garlic)'
        p_generic = re.sub(r'\(.*?\)', '', p_generic).strip()
        if len(p_generic) > 2:
            components.append(p_generic)
    return components

# Apply cleaning
raw_df['ingredient_name_clean'] = raw_df['ingredient_name'].apply(clean_name)
raw_df['is_combination']        = raw_df['ingredient_name_clean'].str.contains(r'[;,/]', regex=True)
raw_df['component_count']       = raw_df['ingredient_name_clean'].apply(
    lambda x: len(re.split(r'[;,/]', x))
)
raw_df['components']            = raw_df['ingredient_name_clean'].apply(
    lambda x: ' | '.join(extract_components(x))
)

# Deduplicate on ingredient_id (the definitive unique key)
clean_df = raw_df.drop_duplicates(subset='ingredient_id').reset_index(drop=True)

print(f'After deduplication: {len(clean_df):,} unique ingredients')
print(f'Single-ingredient   : {(~clean_df["is_combination"]).sum():,}')
print(f'Combination drugs   : {clean_df["is_combination"].sum():,}')
print()
clean_df.head(15)

After deduplication: 2,221 unique ingredients
Single-ingredient   : 1,187
Combination drugs   : 1,034



,ingredient_name,ingredient_id,nafdac_url,page_scraped,ingredient_name_clean,is_combination,component_count,components
0,"2,4-Dichlorobenzyl alcohol , Amylmetacresol ,V...",2198,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,"2, 4-Dichlorobenzyl alcohol, Amylmetacresol, V...",True,4,4-Dichlorobenzyl alcohol | Amylmetacresol | Vi...
1,Abacavir Sulfate,585,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Abacavir Sulfate,False,1,Abacavir
2,Abacavir Sulfate; Lamivudine,1850,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Abacavir Sulfate; Lamivudine,True,2,Abacavir | Lamivudine
3,"ABACAVIR, DOLUTEGRAVIR, LAMIVUDINE",2335,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,"ABACAVIR, DOLUTEGRAVIR, LAMIVUDINE",True,3,ABACAVIR | DOLUTEGRAVIR | LAMIVUDINE
4,Abacavir; Lamivudine,601,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Abacavir; Lamivudine,True,2,Abacavir | Lamivudine
5,Abacavir; Lamivudine; Efavirenz,1467,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Abacavir; Lamivudine; Efavirenz,True,3,Abacavir | Lamivudine | Efavirenz
6,Abacavir; Lamivudine; Zidovudine,602,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Abacavir; Lamivudine; Zidovudine,True,3,Abacavir | Lamivudine | Zidovudine
7,Abiraterone Acetate,1377,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Abiraterone Acetate,False,1,Abiraterone
8,Acarbose,636,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Acarbose,False,1,Acarbose
9,Aceclofenac,74,https://greenbook.nafdac.gov.ng/ingredient/pro...,1,Aceclofenac,False,1,Aceclofenac


In [8]:
# ── Also build a FLAT individual-drug table ───────────────────────────────────
# One row per individual generic drug name (combinations split out)

individual_drugs = []
for _, row in clean_df.iterrows():
    comps = extract_components(row['ingredient_name_clean'])
    for comp in comps:
        individual_drugs.append({
            'generic_name'      : comp.strip(),
            'generic_name_lower': comp.strip().lower(),
            'source_ingredient' : row['ingredient_name_clean'],
            'ingredient_id'     : row['ingredient_id'],
            'is_combination'    : row['is_combination'],
            'nafdac_url'        : row['nafdac_url']
        })

individual_df = (
    pd.DataFrame(individual_drugs)
    .drop_duplicates(subset='generic_name_lower')
    .sort_values('generic_name_lower')
    .reset_index(drop=True)
)

print(f'Unique individual generic drug names: {len(individual_df):,}')
print()
individual_df.head(20)

Unique individual generic drug names: 1,775



,generic_name,generic_name_lower,source_ingredient,ingredient_id,is_combination,nafdac_url
0,& 18) L1 protein vaccine,& 18) l1 protein vaccine,"Human papillomavirus quadrivalent (types 6, 11...",1761,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
1,10 CCID50 Neomycin 15mcg Stabilizer 1M MgCL2,10 ccid50 neomycin 15mcg stabilizer 1m mgcl2,Polio virus grown on primary monkey kidney cul...,1763,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
2,10 CCID50 Type 111&gt,10 ccid50 type 111&gt,Polio virus grown on primary monkey kidney cul...,1763,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
3,17D-204 strain (live,17d-204 strain (live,"Yellow fever virus1, 17D-204 strain (live, att...",1774,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
4,18C,18c,Pneumococcal polysaccharide serotype 1; Pneumo...,2281,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
5,19A,19a,Pneumococcal polysaccharide serotype 1; Pneumo...,2281,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
6,19F,19f,Pneumococcal polysaccharide serotype 1; Pneumo...,2281,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
7,19F and 23F,19f and 23f,"Saccharide for each serotype 1, 5, 6A, 7F, 9V,...",2101,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
8,2- Phenoxyethanol,2- phenoxyethanol,Pneumococcal polysaccharide serotype 1; Pneumo...,2281,True,https://greenbook.nafdac.gov.ng/ingredient/pro...
9,2-Hydroxyethyl salicylate,2-hydroxyethyl salicylate,Methyl salicylate; Ethyl salicylate; 2-Hydroxy...,1338,True,https://greenbook.nafdac.gov.ng/ingredient/pro...


In [10]:
# ── Quick stats & preview ─────────────────────────────────────────────────────

print('=' * 55)
print('  SCRAPE SUMMARY')
print('=' * 55)
print(f'  Total ingredient entries  : {len(clean_df):,}')
print(f'  Single-drug ingredients   : {(~clean_df["is_combination"]).sum():,}')
print(f'  Combination products      : {clean_df["is_combination"].sum():,}')
print(f'  Unique individual generics: {len(individual_df):,}')
print(f'  Pages successfully scraped: {len([p for p in page_summary if p["status"] == "ok"])}')
print(f'  Failed pages              : {failed_pages if failed_pages else "None"}')
print('=' * 55)
print()

print('Most common components in combination products:')
from collections import Counter
all_comps = []
for row in clean_df[clean_df['is_combination']]['components']:
    all_comps.extend([c.strip() for c in row.split('|')])
for name, count in Counter(all_comps).most_common(15):
    print(f'  {count:3d}x  {name}')

  SCRAPE SUMMARY
  Total ingredient entries  : 2,221
  Single-drug ingredients   : 1,187
  Combination products      : 1,034
  Unique individual generics: 1,775
  Pages successfully scraped: 45
  Failed pages              : None

Most common components in combination products:
   78x  Menthol
   64x  Paracetamol
   63x  Nicotinamide
   53x  Folic acid
   46x  Zinc
   45x  Vit B1
   45x  Vitamin B12
   43x  Chlorphenamine
   39x  Vit B2
   39x  Vit B6
   37x  Vitamin A
   37x  Folic Acid
   36x  Vit B12
   35x  Vitamin B1
   35x  Vitamin B6


---
## Step 4 — Save to CSV

We save **two CSV files**:
- `nafdac_ingredients.csv` — full cleaned ingredient list (one row per NAFDAC entry)
- `nafdac_individual_drugs.csv` — flat table of individual generic drug names (combinations split)

Both are saved locally in Colab first, then optionally to Google Drive.

In [11]:
# ── Save locally in Colab (/content/) ────────────────────────────────────────

LOCAL_PATH_1 = '/content/nafdac_ingredients.csv'
LOCAL_PATH_2 = '/content/nafdac_individual_drugs.csv'

clean_df.to_csv(LOCAL_PATH_1, index=False)
individual_df.to_csv(LOCAL_PATH_2, index=False)

print(f'✅ Saved locally:')
print(f'   {LOCAL_PATH_1}  ({len(clean_df):,} rows)')
print(f'   {LOCAL_PATH_2}  ({len(individual_df):,} rows)')

✅ Saved locally:
   /content/nafdac_ingredients.csv  (2,221 rows)
   /content/nafdac_individual_drugs.csv  (1,775 rows)


In [12]:
# ── Download directly to your PC ─────────────────────────────────────────────
# This triggers a browser download for each file

from google.colab import files

print('Downloading files to your PC...')
files.download(LOCAL_PATH_1)
files.download(LOCAL_PATH_2)
print('✅ Downloads triggered — check your browser downloads folder.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloads triggered — check your browser downloads folder.


In [ ]:
# ── Optionally save to Google Drive ──────────────────────────────────────────
# Uncomment this cell if you want the files saved to Drive as well

# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_FOLDER = '/content/drive/MyDrive/ddi_project/data/raw/'

# import os
# os.makedirs(DRIVE_FOLDER, exist_ok=True)

# clean_df.to_csv(DRIVE_FOLDER + 'nafdac_ingredients.csv', index=False)
# individual_df.to_csv(DRIVE_FOLDER + 'nafdac_individual_drugs.csv', index=False)

# print(f'✅ Saved to Google Drive:')
# print(f'   {DRIVE_FOLDER}nafdac_ingredients.csv')
# print(f'   {DRIVE_FOLDER}nafdac_individual_drugs.csv')

---
## Step 5 — Verify the Output

In [ ]:
# ── Reload and verify CSVs ────────────────────────────────────────────────────

verify1 = pd.read_csv(LOCAL_PATH_1)
verify2 = pd.read_csv(LOCAL_PATH_2)

print('nafdac_ingredients.csv')
print(f'  Shape   : {verify1.shape}')
print(f'  Columns : {list(verify1.columns)}')
print()
print(verify1.head(5).to_string())
print()
print('─' * 60)
print('nafdac_individual_drugs.csv')
print(f'  Shape   : {verify2.shape}')
print(f'  Columns : {list(verify2.columns)}')
print()
print(verify2.head(10).to_string())

In [ ]:
# ── Search the dataset ────────────────────────────────────────────────────────
# Quick lookup to verify key Nigerian drugs are present

CHECK_DRUGS = [
    'rifampicin', 'efavirenz', 'artemether', 'lumefantrine',
    'metformin', 'amlodipine', 'ciprofloxacin', 'warfarin',
    'isoniazid', 'nevirapine', 'doxycycline', 'fluconazole'
]

print('🔎 Checking key Nigerian drugs in scraped data:\n')
for drug in CHECK_DRUGS:
    # Search in individual drugs table
    match = verify2[verify2['generic_name_lower'].str.contains(drug, na=False)]
    if len(match) > 0:
        print(f'  ✅ {drug:20s} → found ({len(match)} entry)')
    else:
        # Search in full ingredient name
        match2 = verify1[verify1['ingredient_name_clean'].str.lower().str.contains(drug, na=False)]
        if len(match2) > 0:
            print(f'  ✅ {drug:20s} → found in combo product')
        else:
            print(f'  ⚠️  {drug:20s} → NOT found (may use brand/alternate name)')

print()
print('✅ Scraper complete. Files ready for use in 01_eda.ipynb')